# Dialogue Log Verification System

This notebook verifies and selects the best dialogue from GPT-4o and GPT-5.2 generated seeds using Claude 4.5 Haiku as an independent arbiter with binary pass/fail validation on 5 criteria.

## Execution Flow

The cells execute in 7 sequential steps:
1. Import libraries and set file paths
2. Load GPT-4o and GPT-5.2 CSV datasets
3. Review binary validation system and decision logic
4. Initialize Claude 4.5 Haiku verifier and define verification function
5. Run verification loop (currently limited to first 2 games for testing)
6. Create output DataFrames from verification results with per-criterion tracking
7. Save verified results to CSV files

## Output Files

1. **verified_r1_seeds_combined.csv**: Selected dialogues with source labels
2. **verified_r1_criteria_scores.csv**: Detailed per-criterion pass/fail tracking for analysis

## Step 1: Import Libraries

Import required Python libraries and set file paths.

In [39]:
# Import Required Libraries
import pandas as pd
import json
import os
import re
from typing import Dict, Tuple, Optional
from anthropic import Anthropic
import time

# Initialize paths
DATA_DIR = "."
GPT4O_FILE = "generated_r1_seeds_gpt4o.csv"
GPT52_FILE = "generated_r1_seeds_gpt5_2.csv"
OUTPUT_FILE_COMBINED = "verified_r1_seeds_combined.csv"
OUTPUT_FILE_CRITERIA = "verified_r1_criteria_scores.csv"

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Load Datasets

Load the GPT-4o and GPT-5.2 generated seed datasets.

In [40]:
# Load Generated Seeds from GPT Models
gpt4o_df = pd.read_csv(GPT4O_FILE)
gpt52_df = pd.read_csv(GPT52_FILE)

print(f"GPT-4o dataset loaded: {gpt4o_df.shape[0]} rows, {gpt4o_df.shape[1]} columns")
print(f"GPT-5.2 dataset loaded: {gpt52_df.shape[0]} rows, {gpt52_df.shape[1]} columns")
print(f"\nColumns in GPT-4o: {list(gpt4o_df.columns)}")
print(f"Sample row (first game):")
print(f"  game_id: {gpt4o_df.iloc[0]['game_id']}")
print(f"  player_roles: {gpt4o_df.iloc[0]['player_roles'][:80]}...")
print(f"  matrix_tactic_scale sample: {gpt4o_df.iloc[0]['matrix_tactic_scale'][:100]}...")

GPT-4o dataset loaded: 225 rows, 9 columns
GPT-5.2 dataset loaded: 225 rows, 9 columns

Columns in GPT-4o: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale']
Sample row (first game):
  game_id: G026
  player_roles: {"P1":"Good",
  "P2":"Good",
  "P3":"Evil",
  "P4":"Evil",
  "P5":"Good"}...
  matrix_tactic_scale sample: {"P2": {"row": "Selective/Framing", "col": "Cooperative", "tactic": "Pragmatic silence", "scale": "G...


## Step 3: Binary Multi-Criteria Validation System

**Validation Criteria:**

Claude 4.5 Haiku evaluates each dialogue using **binary pass/fail checks** on 5 criteria:

1. **Player Roles Consistency** (1/0) - Good players use honest tactics, Evil players use deceptive tactics
2. **Public History Alignment** (1/0) - Dialogue contextually appropriate to game state
3. **Matrix Tactic Scale Validity** (1/0) - Correct row, col, tactic, scale, and level
4. **Avalon Gameplay Authenticity** (1/0) - Natural dialogue with strategic plausibility
5. **Dialogue Format Compliance** (1/0) - Exactly 4 speakers, no protagonist, correct format

**Decision Logic:**

1. **Both 5/5 pass** → Claude 4.5 Haiku performs pairwise comparison to break tie
2. **One 5/5, other <5/5** → Select the 5/5 response
3. **At least one 4/5**:
   - One 4/5, other <4/5 → Choose 4/5, **Targeted Correction** (fix only the failing criterion)
   - Both 4/5 → Choose GPT-5.2, **Targeted Correction**
4. **Both ≤3/5** → **Full Custom Generation** (generate from scratch)

## Step 3a: Initialize Claude 4.5 Haiku

Set up the Anthropic client and system prompt.

In [41]:
# Initialize Claude 4.5 Haiku Verifier
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'), max_retries=5)

SYSTEM_PROMPT = """You are an expert verifier for the Avalon social deception game with deep knowledge of deception tactics and game mechanics. Your task is to evaluate pairs of AI-generated dialogue responses, assess their quality, and determine the best outcome: select the superior response as-is, apply targeted corrections to fix specific issues in the better response, or generate an entirely new dialogue if both responses are inadequate."""

print("✓ Claude 4.5 Haiku system prompt initialized")

✓ Claude 4.5 Haiku system prompt initialized


## Step 3b: Define Verification Function

Create the function that compares and evaluates dialogue pairs.

In [ ]:
# Create Verification Function
def verify_dialogue_pair(game_id: str,
                         round_id: str,
                         player_roles: str,
                         protagonist: str,
                         public_history: str,
                         dialogue_gpt4o: str,
                         tactic_scale_gpt4o: str,
                         dialogue_gpt52: str,
                         tactic_scale_gpt52: str) -> Dict:
    """
    Verify both dialogue responses using Claude 4.5 Haiku with binary criteria validation.
    
    Returns: {
        'gpt4o_criteria': {'roles': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gpt4o_total': int (0-5),
        'gpt52_criteria': {'roles': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gpt52_total': int (0-5),
        'recommendation': str,
        'selected_dialogue': str,
        'selected_tactic_scale': str,
        'correction_mode': str (None, Targeted, Full_Custom, Pairwise_Tiebreaker),
        'failed_criterion': str or None,
        'reasoning': str
    }
    """
    
    verification_prompt = f"""You are evaluating two AI-generated Avalon Round 1 discussion logs to determine which is superior. Both responses were generated for the same game context. Your task is to validate both using BINARY PASS/FAIL checks on 5 criteria, then recommend the best outcome: select one response as-is, apply a targeted correction to the better response, or generate a completely new dialogue if both are inadequate.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Public History (up to current round):
{public_history}

---

**RESPONSE A:**

Discussion Log:
{dialogue_gpt4o}

Matrix Tactic Scale:
{tactic_scale_gpt4o}

---

**RESPONSE B:**

Discussion Log:
{dialogue_gpt52}

Matrix Tactic Scale:
{tactic_scale_gpt52}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Player Roles Consistency**: 
   1 = Good players use transparent/honest tactics, Evil players use deceptive tactics
   0 = Role-tactic misalignment or inconsistent behaviors

2. **Public History Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns
   0 = Ignores prior events or contradicts established game state

3. **Matrix Tactic Scale Validity**:
   1 = Correct row (Info Strategy), column (Goal), tactic (1 of 37), scale (GRS/Mach-IV), level (H/M/L)
   0 = Wrong category, invalid tactic, or scale-role mismatch

4. **Avalon Gameplay Authenticity**:
   1 = Natural dialogue with appropriate accusations/defenses, skill-modulated sophistication
   0 = Unnatural, implausible strategy, or skill level doesn't match behavior complexity

5. **Dialogue Format Compliance**:
   1 = Follows "Discussion after Quest X:" format, exactly 4 speakers (excluding protagonist), each speaks once
   1 = Expected format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong format, incorrect speaker count, protagonist speaks, or format violations

**DECISION LOGIC:**

Apply this logic based on how many criteria each response passes:

1. **Both responses 5/5** → Use PAIRWISE_TIEBREAKER
   - Compare both dialogues to determine which is slightly better overall
   - Output RECOMMENDATION: PAIRWISE_TIEBREAKER

2. **One response 5/5, other <5/5** → Choose the 5/5 response
   - Output RECOMMENDATION: RESPONSE_A (if Response A is 5/5)
   - Output RECOMMENDATION: RESPONSE_B (if Response B is 5/5)

3. **At least one response 4/5** → Apply targeted correction
   - If only one is 4/5: choose that one → CORRECTION_A or CORRECTION_B
   - If both are 4/5: default to Response B → CORRECTION_B
   - Output RECOMMENDATION: CORRECTION_A or CORRECTION_B

4. **Both responses ≤3/5** → Generate custom dialogue
   - Output RECOMMENDATION: CUSTOM_GENERATION

**EVALUATION PROCESS:**

1. Evaluate RESPONSE A against all 5 criteria using binary PASS/FAIL checks
2. Evaluate RESPONSE B against all 5 criteria using the same binary checks
3. Count total passes for each response (X/5)
4. Apply the DECISION LOGIC rules based on the pass counts
5. Generate output in the exact format below

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object with the following structure:

```json
{{
  "response_a": {{
    "criteria": {{
      "roles": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "response_b": {{
    "criteria": {{
      "roles": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "recommendation": "RESPONSE_A / RESPONSE_B / PAIRWISE_TIEBREAKER / CORRECTION_A / CORRECTION_B / CUSTOM_GENERATION",
  "reasoning": "2-3 sentences explaining the decision",
  "pairwise_winner": "RESPONSE_A / RESPONSE_B / null",
  "pairwise_reasoning": "1-2 sentences explaining pairwise selection / null",
  "failed_criterion": "Roles / History / Matrix / Authenticity / Format / null",
  "targeted_correction": "Full corrected dialogue / null",
  "custom_dialogue": "Complete new dialogue / null"
}}
```

**Field Explanations:**
- **score**: 1 = PASS, 0 = FAIL
- **explanation**: Brief explanation required for BOTH pass and fail to justify the score
- **total**: Integer count of passed criteria (0-5), NOT a fraction string
- **recommendation**: One of: "RESPONSE_A", "RESPONSE_B", "PAIRWISE_TIEBREAKER", "CORRECTION_A", "CORRECTION_B", "CUSTOM_GENERATION"
- **pairwise_winner**: Only if recommendation is PAIRWISE_TIEBREAKER (both 5/5), set to "RESPONSE_A" or "RESPONSE_B"
- **pairwise_reasoning**: Only if pairwise_winner is set, 1-2 sentences
- **failed_criterion**: Only if recommendation is CORRECTION_A or CORRECTION_B (4/5 needs fix), set to: "Roles", "History", "Matrix", "Authenticity", or "Format"
- **targeted_correction**: Only if failed_criterion is set, provide full corrected dialogue
- **custom_dialogue**: Only if recommendation is CUSTOM_GENERATION (both ≤3/5), provide complete new dialogue in format:
  ```
  Discussion after Quest 1:
  PlayerX: [dialogue]
  PlayerY: [dialogue]
  PlayerZ: [dialogue]
  PlayerW: [dialogue]
  ```

**IMPORTANT:** Return ONLY the JSON object, no markdown code blocks, no extra text."""
    
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": verification_prompt}
            ]
        )
        
        response_text = response.content[0].text
        
        # DEBUG: Print raw response to diagnose parsing issues
        print(f"\n{'='*80}")
        print(f"RAW CLAUDE RESPONSE FOR {game_id}:")
        print(f"{'='*80}")
        print(response_text)
        print(f"{'='*80}\n")
        
        # Parse JSON response
        try:
            # Remove markdown code blocks if present
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                # Remove opening ``` line
                lines = lines[1:]
                # Find and remove closing ```
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)
            
            # Parse JSON
            data = json.loads(clean_text)
            
            # Build result dictionary from JSON
            result = {
                'gpt4o_criteria': {
                    'roles': data['response_a']['criteria']['roles']['score'] == 1,
                    'history': data['response_a']['criteria']['history']['score'] == 1,
                    'matrix': data['response_a']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_a']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_a']['criteria']['format']['score'] == 1
                },
                'gpt4o_total': int(data['response_a']['total']) if isinstance(data['response_a']['total'], int) else int(str(data['response_a']['total']).split('/')[0]),
                'gpt4o_criteria_explanations': {
                    criterion: data['response_a']['criteria'][criterion].get('explanation', '')
                    for criterion in ['roles', 'history', 'matrix', 'authenticity', 'format']
                },
                'gpt52_criteria': {
                    'roles': data['response_b']['criteria']['roles']['score'] == 1,
                    'history': data['response_b']['criteria']['history']['score'] == 1,
                    'matrix': data['response_b']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_b']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_b']['criteria']['format']['score'] == 1
                },
                'gpt52_total': int(data['response_b']['total']) if isinstance(data['response_b']['total'], int) else int(str(data['response_b']['total']).split('/')[0]),
                'gpt52_criteria_explanations': {
                    criterion: data['response_b']['criteria'][criterion].get('explanation', '')
                    for criterion in ['roles', 'history', 'matrix', 'authenticity', 'format']
                },
                'recommendation': data['recommendation'],
                'reasoning': data['reasoning'],
                'pairwise_winner': data.get('pairwise_winner'),
                'pairwise_reasoning': data.get('pairwise_reasoning', ''),
                'failed_criterion': data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'custom_dialogue': data.get('custom_dialogue'),
                'selected_dialogue': None,
                'selected_tactic_scale': None,
                'correction_mode': None
            }
            
            # Determine final selection based on scores
            s1 = result['gpt4o_total']
            s2 = result['gpt52_total']
            
            if s1 == 5 and s2 == 5:
                # Both 5/5 - use pairwise tiebreaker
                if result['pairwise_winner'] and 'RESPONSE_A' in result['pairwise_winner'].upper():
                    result['recommendation'] = 'RESPONSE_A'
                    result['selected_dialogue'] = dialogue_gpt4o
                    result['selected_tactic_scale'] = tactic_scale_gpt4o
                else:
                    result['recommendation'] = 'RESPONSE_B'
                    result['selected_dialogue'] = dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = 'Pairwise_Tiebreaker'
                
            elif s1 == 5:
                # Only Response A is 5/5
                result['recommendation'] = 'RESPONSE_A'
                result['selected_dialogue'] = dialogue_gpt4o
                result['selected_tactic_scale'] = tactic_scale_gpt4o
                result['correction_mode'] = None
                
            elif s2 == 5:
                # Only Response B is 5/5
                result['recommendation'] = 'RESPONSE_B'
                result['selected_dialogue'] = dialogue_gpt52
                result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = None
                
            elif s1 == 4 or s2 == 4:
                # At least one 4/5 - choose 4/5 or default to Response B if both 4/5
                if s1 == 4 and s2 == 4:
                    # Both 4/5 - default to Response B
                    result['recommendation'] = 'CORRECTION_B'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                elif s1 == 4:
                    result['recommendation'] = 'CORRECTION_A'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gpt4o
                    result['selected_tactic_scale'] = tactic_scale_gpt4o
                else:  # s2 == 4
                    result['recommendation'] = 'CORRECTION_B'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = 'Targeted'
                
            else:
                # Both <=3/5 - full custom generation
                result['correction_mode'] = 'Full_Custom'
                result['selected_dialogue'] = result['custom_dialogue']
                result['selected_tactic_scale'] = tactic_scale_gpt52  # Use Response B tactic as base
                result['recommendation'] = 'CUSTOM_GENERATION'
            
            return result, response_text
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            print(f"Cleaned text: {clean_text[:500]}...")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing JSON for {game_id}: {e}")
            return None, str(e)
    except Exception as e:
        print(f"Error verifying game {game_id}: {str(e)}")
        return None, str(e)

## Step 4: Run Verification Loop

Process games and collect verification results.

In [ ]:
# Helper function to format matrix tactic scale with pretty-printing
def format_matrix_tactic_scale(json_str: str) -> str:
    """Format JSON with linebreaks for readability"""
    try:
        data = json.loads(json_str)
        # Reconstruct with pretty-printing: 2-space indent
        return json.dumps(data, indent=2)
    except:
        return json_str

# Compare and Validate Dialogue Rows
# Initialize results storage
verified_results = []
criteria_results = []
verification_stats = {
    'total_games': 0,
    'response_1_selected': 0,
    'response_2_selected': 0,
    'targeted_correction_1': 0,
    'targeted_correction_2': 0,
    'pairwise_tiebreaker': 0,
    'full_custom_generation': 0,
    'errors': 0
}

# Process first 2 games as a sample/test run. Replace 2 with 225 or len(gpt4o_df)
TEST_LIMIT = len(gpt4o_df)
print(f"Starting verification of {min(TEST_LIMIT, len(gpt4o_df))} games...")
print("=" * 80)

for idx in range(min(TEST_LIMIT, len(gpt4o_df))):
    game_id = gpt4o_df.iloc[idx]['game_id']
    round_id = gpt4o_df.iloc[idx]['round_id']
    score_id = f"{game_id}/{round_id}"
    
    # Extract fields from both datasets
    player_roles = gpt4o_df.iloc[idx]['player_roles']
    protagonist = gpt4o_df.iloc[idx]['role_id']  # Protagonist who doesn't speak
    public_history = gpt4o_df.iloc[idx]['public_history']
    
    dialogue_gpt4o = gpt4o_df.iloc[idx]['discussion_log']
    tactic_scale_gpt4o = gpt4o_df.iloc[idx]['matrix_tactic_scale']
    
    dialogue_gpt52 = gpt52_df.iloc[idx]['discussion_log']
    tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']
    
    # Verify the pair
    print(f"\n[{idx+1}/{min(TEST_LIMIT, len(gpt4o_df))}] Verifying game {score_id}...")
    result, raw_response = verify_dialogue_pair(
        game_id, round_id, player_roles, protagonist, public_history,
        dialogue_gpt4o, tactic_scale_gpt4o,
        dialogue_gpt52, tactic_scale_gpt52
    )
    
    if result is None:
        print(f"  ✗ Error: {raw_response}")
        verification_stats['errors'] += 1
        continue
    
    verification_stats['total_games'] += 1
    
    # Determine LLM source and label for output (map Response A/B back to actual model names)
    if result['recommendation'] == 'RESPONSE_A':
        llm_used = 'GPT-4o'
        selected_row = gpt4o_df.iloc[idx].copy()
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt4o)
        verification_stats['response_1_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_A':
        llm_used = 'GPT-4o-Claude-4.5'
        selected_row = gpt4o_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt4o)
        verification_stats['targeted_correction_1'] += 1
    elif result['recommendation'] == 'RESPONSE_B':
        llm_used = 'GPT-5.2'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['response_2_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_B':
        llm_used = 'GPT-5.2-Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['targeted_correction_2'] += 1
    else:  # CUSTOM_GENERATION
        llm_used = 'Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['full_custom_generation'] += 1
    
    # Track pairwise tiebreakers separately
    if result['correction_mode'] == 'Pairwise_Tiebreaker':
        verification_stats['pairwise_tiebreaker'] += 1
    
    # Update matrix tactic scale in selected row
    selected_row['matrix_tactic_scale'] = selected_tactic_scale
    
    # Add LLM_used and Claude reasoning columns
    selected_row['LLM_used'] = llm_used
    selected_row['Claude_Reasoning'] = result['reasoning']
    verified_results.append(selected_row)
    
    # Create detailed criteria record for CSV
    def format_check(val):
        return 1 if val else 0
    
    criteria_record = {
        'ID': score_id,
        'GPT4o_Roles': format_check(result['gpt4o_criteria']['roles']),
        'GPT4o_History': format_check(result['gpt4o_criteria']['history']),
        'GPT4o_Matrix': format_check(result['gpt4o_criteria']['matrix']),
        'GPT4o_Authenticity': format_check(result['gpt4o_criteria']['authenticity']),
        'GPT4o_Format': format_check(result['gpt4o_criteria']['format']),
        'GPT4o_Total': f"{result['gpt4o_total']} of 5",
        'GPT52_Roles': format_check(result['gpt52_criteria']['roles']),
        'GPT52_History': format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix': format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity': format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format': format_check(result['gpt52_criteria']['format']),
        'GPT52_Total': f"{result['gpt52_total']} of 5",
        'Selected_LLM': llm_used,
        'Correction_Mode': result['correction_mode'] or 'None',
        'Claude_Reasoning': result['reasoning'],
        'GPT4o_Criteria_Explanations': json.dumps(result['gpt4o_criteria_explanations']),
        'GPT52_Criteria_Explanations': json.dumps(result['gpt52_criteria_explanations']),
        'Claude_Response': raw_response  # Store full raw response for debugging
    }
    criteria_results.append(criteria_record)
    
    # Print summary
    print(f"  ✓ GPT-4o: {result['gpt4o_total']}/5 pass ({', '.join([k for k,v in result['gpt4o_criteria'].items() if v])})")
    print(f"  ✓ GPT-5.2: {result['gpt52_total']}/5 pass ({', '.join([k for k,v in result['gpt52_criteria'].items() if v])})")
    print(f"  ✓ Selected: {llm_used}")
    if result['correction_mode']:
        print(f"  ✓ Correction Mode: {result['correction_mode']}")
    print(f"  ✓ Reason: {result['reasoning'][:100]}...")
    
    # Debug output for custom generation
    if result['correction_mode'] == 'Full_Custom':
        if result['selected_dialogue']:
            print(f"  📝 Custom dialogue preview: {result['selected_dialogue'][:150]}...")
        else:
            print(f"  ⚠️  WARNING: Custom dialogue is empty!")
    
    # Rate limit to avoid API throttling
    time.sleep(1)

print("\n" + "=" * 80)
print(f"Verification Summary (first {TEST_LIMIT} games):")
print(f"  Total verified: {verification_stats['total_games']}")
print(f"  GPT-4o selected (original): {verification_stats['response_1_selected']}")
print(f"  GPT-5.2 selected (original): {verification_stats['response_2_selected']}")
print(f"  Pairwise tiebreaker used: {verification_stats['pairwise_tiebreaker']}")
print(f"  GPT-4o + Targeted correction: {verification_stats['targeted_correction_1']}")
print(f"  GPT-5.2 + Targeted correction: {verification_stats['targeted_correction_2']}")
print(f"  Claude 4.5 full custom generation: {verification_stats['full_custom_generation']}")
print(f"  Errors: {verification_stats['errors']}")

Starting verification of 2 games...

[1/2] Verifying game G026/1...

RAW CLAUDE RESPONSE FOR G026:
```json
{
  "response_a": {
    "criteria": {
      "roles": {
        "score": 1,
        "explanation": "Good players (P2, P5) use transparent/cooperative tactics appropriate to their alignment. Evil players (P3, P4) employ deceptive tactics (counterfactual falsehoods, opportunistic overstatement) consistent with their roles."
      },
      "history": {
        "score": 1,
        "explanation": "Dialogue appropriately references Quest 1 passing with team P1+P3, universal approval voting, and builds naturally on this outcome. All comments contextually fit the public game state."
      },
      "matrix": {
        "score": 1,
        "explanation": "All tactic assignments are valid: P2's pragmatic silence (GRS/Selective), P3's bold falsehoods (Mach-IV/Counterfactual), P4's overstated contribution (Mach-IV/Transparent), P5's perspective-taking (GRS/Transparent). Scales and levels appropr

## Step 5: Create Output DataFrames

Convert verification results into structured DataFrames.

In [44]:
# Create Output DataFrames
# Create DataFrame from verified results (full data)
if verified_results:
    verified_df = pd.DataFrame(verified_results)
    
    # Ensure LLM_used is last column
    cols = verified_df.columns.tolist()
    cols = [col for col in cols if col != 'LLM_used'] + ['LLM_used']
    verified_df = verified_df[cols]
    
    print(f"\n✓ Verified DataFrame (combined) shape: {verified_df.shape}")
    print(f"✓ Columns: {list(verified_df.columns)}")
    print(f"\nFirst verified row (LLM_used column):")
    print(verified_df[['game_id', 'round_id', 'LLM_used']].head())
else:
    print("No results to create verified output CSV")
    verified_df = None

# Create DataFrame from criteria results (detailed per-criterion tracking)
if criteria_results:
    criteria_df = pd.DataFrame(criteria_results)
    
    print(f"\n✓ Criteria DataFrame shape: {criteria_df.shape}")
    print(f"✓ Columns: {list(criteria_df.columns)}")
    print(f"\nFirst criteria records:")
    print(criteria_df.head())
    
    # Calculate aggregate statistics
    print(f"\n{'='*60}")
    print("AGGREGATE STATISTICS (for paper analysis):")
    print(f"{'='*60}")
    
    # Helper function to count passes
    def count_passes(df, prefix):
        roles = (df[f'{prefix}_Roles'] == 1).sum()
        history = (df[f'{prefix}_History'] == 1).sum()
        matrix = (df[f'{prefix}_Matrix'] == 1).sum()
        authenticity = (df[f'{prefix}_Authenticity'] == 1).sum()
        format_check = (df[f'{prefix}_Format'] == 1).sum()
        total = len(df)
        return {
            'Roles': f"{roles}/{total} ({roles/total*100:.1f}%)",
            'History': f"{history}/{total} ({history/total*100:.1f}%)",
            'Matrix': f"{matrix}/{total} ({matrix/total*100:.1f}%)",
            'Authenticity': f"{authenticity}/{total} ({authenticity/total*100:.1f}%)",
            'Format': f"{format_check}/{total} ({format_check/total*100:.1f}%)"
        }
    
    gpt4o_stats = count_passes(criteria_df, 'GPT4o')
    gpt52_stats = count_passes(criteria_df, 'GPT52')
    
    print("\nGPT-4o Pass Rates:")
    for criterion, stat in gpt4o_stats.items():
        print(f"  {criterion:15s}: {stat}")
    
    print("\nGPT-5.2 Pass Rates:")
    for criterion, stat in gpt52_stats.items():
        print(f"  {criterion:15s}: {stat}")
else:
    print("No results to create criteria output CSV")
    criteria_df = None


✓ Verified DataFrame (combined) shape: (2, 11)
✓ Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'Claude_Reasoning', 'LLM_used']

First verified row (LLM_used column):
  game_id  round_id LLM_used
0    G026         1  GPT-5.2
1    G027         1  GPT-5.2

✓ Criteria DataFrame shape: (2, 19)
✓ Columns: ['ID', 'GPT4o_Roles', 'GPT4o_History', 'GPT4o_Matrix', 'GPT4o_Authenticity', 'GPT4o_Format', 'GPT4o_Total', 'GPT52_Roles', 'GPT52_History', 'GPT52_Matrix', 'GPT52_Authenticity', 'GPT52_Format', 'GPT52_Total', 'Selected_LLM', 'Correction_Mode', 'Claude_Reasoning', 'GPT4o_Criteria_Explanations', 'GPT52_Criteria_Explanations', 'Claude_Response']

First criteria records:
       ID  GPT4o_Roles  GPT4o_History  GPT4o_Matrix  GPT4o_Authenticity  \
0  G026/1            1              1             1                   0   
1  G027/1            0              1             1               

## Step 6: Output Files

Files saved by the verification process:

1. **verified_r1_seeds_combined.csv**: Full dialogue rows with columns:
   - All original columns from input CSVs
   - `LLM_used`: Source indicator (GPT-4o, GPT-5.2, GPT-4o-Claude-4.5, GPT-5.2-Claude-4.5, or Claude-4.5)
   - `Claude_Reasoning`: Claude's explanation for the selection decision

2. **verified_r1_criteria_scores.csv**: Detailed per-criterion validation tracking with columns:
   - `ID`: Game/Round identifier
   - `GPT4o_Roles`, `GPT4o_History`, `GPT4o_Matrix`, `GPT4o_Authenticity`, `GPT4o_Format`: Binary checks (1 = pass, 0 = fail)
   - `GPT4o_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `GPT52_Roles`, `GPT52_History`, `GPT52_Matrix`, `GPT52_Authenticity`, `GPT52_Format`: Binary checks (1 = pass, 0 = fail)
   - `GPT52_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `Selected_LLM`: Which model's output was used
   - `Correction_Mode`: None, Pairwise_Tiebreaker, Targeted, or Full_Custom
   - `Claude_Reasoning`: Claude's explanation for the decision
   - `GPT4o_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)
   - `GPT52_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)

**For Paper Analysis:** The criteria CSV enables reporting like: "GPT-4o achieved 95% format compliance (214/225 games)" and identifying which criteria are most challenging for each model. Criteria explanations provide insights into why each criterion passed or failed.

In [45]:
# Save Verified Results
if verified_df is not None and criteria_df is not None:
    # Save combined verified results
    verified_df.to_csv(OUTPUT_FILE_COMBINED, index=False)
    print(f"✓ Verified results saved to {OUTPUT_FILE_COMBINED}")
    print(f"  Total rows: {len(verified_df)}")
    
    # Save criteria tracking with detailed per-criterion scores
    criteria_df.to_csv(OUTPUT_FILE_CRITERIA, index=False)
    print(f"\n✓ Criteria tracking saved to {OUTPUT_FILE_CRITERIA}")
    print(f"  Total records: {len(criteria_df)}")
    
    # Display samples
    print(f"\nSample of verified results (LLM_used distribution):")
    print(verified_df['LLM_used'].value_counts())
    
    print(f"\nSample of criteria records (first 5):")
    print(criteria_df.head(5))
    
    # Display correction mode distribution
    print(f"\nCorrection mode distribution:")
    print(criteria_df['Correction_Mode'].value_counts())
else:
    print("No verified results to save")

✓ Verified results saved to verified_r1_seeds_combined.csv
  Total rows: 2

✓ Criteria tracking saved to verified_r1_criteria_scores.csv
  Total records: 2

Sample of verified results (LLM_used distribution):
LLM_used
GPT-5.2    2
Name: count, dtype: int64

Sample of criteria records (first 5):
       ID  GPT4o_Roles  GPT4o_History  GPT4o_Matrix  GPT4o_Authenticity  \
0  G026/1            1              1             1                   0   
1  G027/1            0              1             1                   0   

   GPT4o_Format GPT4o_Total  GPT52_Roles  GPT52_History  GPT52_Matrix  \
0             1      4 of 5            1              1             1   
1             1      3 of 5            1              1             1   

   GPT52_Authenticity  GPT52_Format GPT52_Total Selected_LLM Correction_Mode  \
0                   1             1      5 of 5      GPT-5.2            None   
1                   1             1      5 of 5      GPT-5.2            None   

                 

## Next Steps: Full Verification

After testing this notebook with the first 2 games and reviewing the results:

1. **Review Sample Results**: Check that binary validation logic works correctly
2. **Check Criteria Distribution**: Examine which criteria pass/fail most frequently
3. **Validate Corrections**: 
   - Check "Targeted" corrections properly fix only the failing criterion
   - Verify "Full_Custom" generations when both models score ≤3/5
   - Confirm "Pairwise_Tiebreaker" selections when both score 5/5
4. **Verify Decision Logic**: Ensure the system correctly handles all scenarios:
   - Both 5/5 → Pairwise comparison
   - One 5/5, other <5 → Selects 5/5
   - At least one 4/5 → Targeted correction (defaults to GPT-5.2 if both 4/5)
   - Both ≤3/5 → Full custom generation
5. **Scale Up**: Change `TEST_LIMIT = 2` to `TEST_LIMIT = len(gpt4o_df)` to process all 225 games
6. **Analyze Results**: Use criteria CSV to calculate per-criterion pass rates for your paper
7. **Final Merge**: Combine verified results with 25 manually generated seeds for complete 250-seed dataset

**Important Notes:**
- Current test uses TEST_LIMIT = 2 to validate the system
- Full verification of 225 games will take ~4-5 minutes (1 sec delay + longer max_tokens)
- Per-criterion tracking enables detailed analysis for paper (e.g., "Format: 98% pass rate")